# Error analysis: which randomization buckets correlate with failures?

Runs locally (CPU is fine) against a trained checkpoint and the synthetic
test set. Joins each test image's per-image mIoU with the randomization
parameters `data_gen` recorded for it in `metadata.jsonl` — the whole
point of writing every randomization parameter to metadata in Phase 1 was
to make this join possible without re-deriving anything.

Output of this notebook is exactly the evidence behind the top-level
README's "Limitations & sim-to-real gaps" section — it should say *which*
conditions the model struggles with, not just that it sometimes fails.

In [ ]:
import json
from pathlib import Path

import jsonlines
import numpy as np
import pandas as pd
import torch

from training.dataset import NUM_CLASSES, SegmentationDataset
from training.metrics import ConfusionMatrixTracker
from training.model import build_model

DATA_DIR = Path('data_gen/output')
CHECKPOINT = Path('training/checkpoints/best.pt')
MODEL_NAME = 'deeplabv3_mobilenet'
IMAGE_SIZE = 512

In [ ]:
model = build_model(MODEL_NAME, num_classes=NUM_CLASSES, pretrained_backbone=False)
model.load_state_dict(torch.load(CHECKPOINT, map_location='cpu'))
model.eval()

splits = json.loads((DATA_DIR / 'splits.json').read_text())
test_indices = set(splits['test'])

with jsonlines.open(DATA_DIR / 'metadata.jsonl') as reader:
    metadata_by_index = {row['index']: row for row in reader if row['index'] in test_indices}

In [ ]:
# Per-image mIoU, not the aggregate confusion-matrix mIoU benchmark.py
# reports — we need one number per image to join against its own
# randomization params.
dataset = SegmentationDataset(DATA_DIR, 'test', image_size=IMAGE_SIZE, augment=False)
records = []
with torch.no_grad():
    for i in range(len(dataset)):
        image, mask = dataset[i]
        row = dataset.rows[i]
        logits = model(image.unsqueeze(0))
        pred = logits.argmax(dim=1)

        tracker = ConfusionMatrixTracker(NUM_CLASSES)
        tracker.update(pred, mask.unsqueeze(0))

        params = metadata_by_index[row['index']]
        records.append({
            'index': row['index'],
            'miou': tracker.mean_iou(),
            'defect_iou': tracker.per_class_iou()[2],
            'num_defects': len(params['defect_patches']),
            'glare_enabled': params['glare_enabled'],
            'motion_blur_enabled': params['motion_blur_enabled'],
            'use_hdri': params['use_hdri'],
            'sensor_noise_std': params['sensor_noise_std'],
            'camera_elevation_deg': params['camera_elevation_deg'],
            'camera_distance_m': params['camera_distance_m'],
            'background_clutter_count': params['background_clutter_count'],
        })

df = pd.DataFrame.from_records(records)
df.describe()

In [ ]:
# Boolean/categorical buckets: does the condition being present correlate with lower mIoU?
for column in ['glare_enabled', 'motion_blur_enabled', 'use_hdri']:
    print(df.groupby(column)['miou'].agg(['mean', 'count']))
    print()

In [ ]:
# Continuous params: bucket into quartiles and look for a trend.
for column in ['sensor_noise_std', 'camera_elevation_deg', 'camera_distance_m', 'num_defects']:
    try:
        df[f'{column}_bucket'] = pd.qcut(df[column], q=4, duplicates='drop')
    except ValueError:
        continue  # not enough distinct values to form quartiles on a small test set
    print(df.groupby(f'{column}_bucket')['miou'].agg(['mean', 'count']))
    print()

In [ ]:
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

worst = df.nsmallest(6, 'miou')
print('Worst-performing test images (lowest per-image mIoU):')
worst[['index', 'miou', 'glare_enabled', 'motion_blur_enabled', 'sensor_noise_std', 'camera_elevation_deg']]

## Writing this up for the README

Whatever buckets show a real (not noise-level, check the `count` column —
a small test set makes single-digit-count buckets unreliable) mIoU gap
here is exactly what belongs in the top-level README's "Limitations &
sim-to-real gaps" section, phrased as a concrete next step rather than a
vague caveat — e.g. "glare-enabled images score N points lower mIoU;
next dataset run should either raise `effects.glare_probability` in
`data_gen/configs/randomization.yaml` to see more of these during training,
or add a glare-specific augmentation."